# Lab 5, part 2: the most likely path (extra credit)

`lab5_localization.ipynb` answered one question, for many time steps: given
everything observed so far, where is the robot now?

However, writing those answers down in order does not necessarily give a likely *path*. This is what we will work on here.

In [1]:
# PROVIDED
import math

import numpy as np
import pandas as pd

from lab5kit.world import DIRECTIONS, Maze
from lab5kit.measures import most_likely
from lab5kit import experiments as experiment
from lab5kit import viz
from lab5kit.checks import (check_all, check_viterbi_path,
                            fixture_trajectory, path_log_probability)

world = Maze()
print(world)

Maze(30 free cells, epsilon=0.1, p_move=0.8)


## Part 0: paste your filter from part a.

Paste your `sensor_likelihood`, `predict`, `update` and `filter_sequence`
from `lab5_localization.ipynb` into the next cell, exactly as you wrote them. 

In [ ]:
# Paste your four functions from lab5_localization.ipynb here.

In [ ]:
# PROVIDED
check_all(sensor_likelihood, predict, update, filter_sequence);

## Part 1: the route that cannot have happened

The cell below filters the run from Part 5 of the core notebook again. At
each time step it takes the cell the belief rates highest, using
`most_likely`, and strings those cells together.

In [ ]:
# PROVIDED
trajectory = fixture_trajectory()
beliefs = filter_sequence(world, world.uniform_belief(),
                          trajectory.actions, trajectory.readings)
step_by_step = [most_likely(belief) for belief in beliefs]

print("t   true cell   best guess")
for t, (truth, guess) in enumerate(zip(trajectory.cells, step_by_step)):
    flag = "" if truth == guess else "   <- different"
    print(f"{t:<4}{str(truth):<12}{str(guess)}{flag}")

Eight of the thirteen are correct, which is reasonable from a standing
start. Now consider the sequence as a **route**, rather than as thirteen
separate answers.

In [ ]:
# PROVIDED
viz.show_paths(world, trajectory, [step_by_step], ["per-step best guess"]);

In [ ]:
# PROVIDED
broken = experiment.illegal_steps(world, step_by_step)
print(f"{len(broken)} step(s) of this route no single action could produce:")
for t, here, there in broken:
    print(f"   t = {t}: {here} -> {there}, "
          f"action '{trajectory.actions[t]}'")

score = path_log_probability(world, world.uniform_belief(),
                             trajectory.actions, trajectory.readings,
                             step_by_step)
print(f"\nlog P(this route and these readings) = {score}")

Three of the twelve steps are impossible. The first step is the worst: between `t = 1` and `t = 2` the route jumps from the top-left corridor
to the bottom-right corner, which one step south cannot do. The probability of the whole route is therefore actually zero. The log probability that was returned is therefore negative infinity.

This is not to say that anything has gone wrong here. Each of those thirteen answers is the best possible answer to the question it was asked. The difficulty is that

$$\arg\max_{s} P(s_t = s \mid \text{everything}) \quad \text{for each } t$$

and

$$\arg\max_{s_0 \ldots s_T} P(s_0 \ldots s_T \mid \text{everything})$$

are different questions. The first asks about one time step at a time, and
is never required to make its answers consistent with one another. The
second asks about the whole route at once, and consistency is forced upon
it: a route scores above zero only if every step in it is a step the robot
could have taken.

When what you want is a trajectory, such as the path a vehicle drove, or a sequence of words in a message, we have to ask the second type of question.

## Part 2: scoring a whole route

The score of a route $s_0, s_1, \ldots, s_T$, given the actions and the
readings, is a single long product. It combines how likely the first cell
was with, for each step, the probability that the move went that way and the
probability that the sonar produced the reading it did.

$$P(s_0) \, P(z_0 \mid s_0) \prod_{t=1}^{T} P(s_t \mid s_{t-1}, a_{t-1}) \, P(z_t \mid s_t)$$

Every factor is something you already have: `belief[cell]`,
`sensor_likelihood`, and `world.transition`. `path_log_probability` in
`lab5kit.checks` computes exactly that product in logarithms, and you have
just seen it return $-\infty$ for a route with a gap in it.

### Why logarithms

Thirteen steps multiply about thirty factors, each of them well under 1. The
product is around $10^{-13}$ here, and over a few hundred steps it
underflows to 0 and takes every route's score with it. Adding logarithms
does the same job and cannot underflow:

$$\log P = \log P(s_0) + \log P(z_0 \mid s_0) + \sum_{t} \big[ \log P(s_t \mid s_{t-1}, a_{t-1}) + \log P(z_t \mid s_t) \big]$$

`math.log(0)` raises an error, so a factor of zero has to be handled
separately. The usual approach is a small helper that returns `-math.inf`
instead.

### Why you cannot try every route

Thirty cells over thirteen time steps gives $30^{13}$ routes, which is about
$10^{19}$. Enumerating them is not an option.

But there is hope: The key is to realize that because this is a Markov chain, we don't need to enumerate everything. The best route ending
in cell $c$ at time $t$ depends only on the score of the best route ending in
each possible predecessor, and not on how that predecessor was reached. We
can therefore keep, for every cell, the score of the best route that ends
there, which is thirty numbers, and advance one step at a time. Thirty
numbers times thirteen steps times thirty predecessors is about twelve
thousand arithmetic operations.

This is the **Viterbi algorithm**, and it is the same dynamic programming
you have met before. Lab 2's hill climbing kept one state and Lab 3's
minimax kept one value per position. This keeps one best score so far per
cell ($P_{t,s}$), together with a record of the predecessor that generated it ($Q_{t,s}$).

### Task 5

The algorithm has three parts:

1. **Start.** For every cell, compute the score of the initial state, $\log P(s_0) + \log P(z_0 \mid s_0)$.
2. **Step.** For each action and the reading that follows it, compute for every cell the best score of any route ending in that cell. That is the maximum over predecessors of the predecessor's score plus the log of the move's probability. Add this cell's reading likelihood afterwards, and record which predecessor won.
3. **Read it back.** Take the best-scoring cell at the last time step and follow the pointers backwards to the start.

The path comes out in reverse order, so reverse it before returning it.

In [ ]:
def viterbi_path(world, belief, actions, readings):
    """The single most likely sequence of cells the robot could have visited.

    Args:
        world: The Maze the filter believes in.
        belief: The belief before any reading, as a prior on the first cell.
        actions: The actions taken, in order. Length T.
        readings: The sonar readings, in order. Length T + 1.

    Returns:
        A list of T + 1 cells, earliest first, maximizing
        P(cells and readings | actions). Consecutive cells are always one
        action apart, so the path is one the robot could really have taken.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_viterbi_path(viterbi_path)

## Part 3: comparing the per-step best guess with viterbi

In [ ]:
# PROVIDED
best_path = viterbi_path(world, world.uniform_belief(),
                         trajectory.actions, trajectory.readings)

rows = []
for name, path in [("per-step best guess", step_by_step),
                   ("most likely path", best_path)]:
    rows.append({
        "method": name,
        "cells right": sum(1 for a, b in zip(path, trajectory.cells) if a == b),
        "agrees with the truth": f"{experiment.agreement(path, trajectory.cells):.0%}",
        "impossible steps": len(experiment.illegal_steps(world, path)),
        "log probability": path_log_probability(
            world, world.uniform_belief(), trajectory.actions,
            trajectory.readings, path),
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# PROVIDED
viz.show_paths(world, trajectory, [step_by_step, best_path],
               ["per-step best guess", "most likely path"]);

In [ ]:
# PROVIDED
tied = sorted(cell for cell, probability in beliefs[0].items()
              if abs(probability - max(beliefs[0].values())) < 1e-12)
print(f"at t = 0, {len(tied)} cells are tied for most likely at "
      f"{max(beliefs[0].values()):.4f} each:")
print(f"   {tied}")
print(f"\nthe filter's tie-break:      {step_by_step[0]}")
print(f"where the best path starts:  {best_path[0]}")
print(f"where the robot really was:  {trajectory.cells[0]}")

---

### What you built

You built a filter that tracks a hidden variable through time, and an algorithm that recovers its entire history. Between them they cover most of what there is to say about hidden Markov models, which were the state of the art in speech
recognition for thirty years and are still widely used in a variety of fields.

For example, a food delivery robot may use filtering to determine its most probable current location. After it made its journey, it may use Viterbi to calculate its most likely trajectory.